# 🎙️ Unlimited Voice Cloning — No Key Required
Personal use version — MIT License

**Run cells in order: Cell 1 → Cell 2 → Cell 3 → Cell 4**

In [ ]:
%%bash
set -e

cd /content

# Download micromamba
curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xvj bin/micromamba

# Create Python 3.11 env
./bin/micromamba create -y -n cb311 -c conda-forge python=3.11 pip

echo '✅ micromamba ready'
echo '✅ env created: cb311'

In [ ]:
%%bash
set -euo pipefail

cd /content
MICROMAMBA='/content/bin/micromamba'

ts() { date +'[%Y-%m-%d %H:%M:%S]'; }

echo "$(ts) Upgrading pip..."
"$MICROMAMBA" run -n cb311 python -m pip install -U pip setuptools wheel --progress-bar on

echo "$(ts) Installing PyTorch 2.5.1 (CUDA 12.1)..."
"$MICROMAMBA" run -n cb311 pip install \
  --progress-bar on \
  torch==2.5.1+cu121 torchaudio==2.5.1+cu121 torchvision==0.20.1+cu121 \
  --index-url https://download.pytorch.org/whl/cu121

echo "$(ts) Installing Chatterbox..."
"$MICROMAMBA" run -n cb311 pip uninstall -y chatterbox-tts chatterbox || true
"$MICROMAMBA" run -n cb311 pip install \
  --no-cache-dir --upgrade \
  --progress-bar on \
  'chatterbox-tts @ git+https://github.com/devnen/chatterbox-v2.git@master'

echo "$(ts) Installing s3tokenizer + onnx..."
"$MICROMAMBA" run -n cb311 pip install --no-deps s3tokenizer==0.3.0 onnx==1.16.0

echo "$(ts) Fixing protobuf..."
"$MICROMAMBA" run -n cb311 pip install --no-deps --force-reinstall 'protobuf>=4.25.0'

echo "$(ts) ✅ Installation complete!"

In [ ]:
import os, subprocess, socket
from pathlib import Path

PORT = 8004
REPO_DIR = '/content/Unlimited-Voice-Cloning'
LOG_STDOUT = '/content/chatterbox_server_stdout.log'

def sh(cmd, check=False):
    return subprocess.run(['bash', '-lc', cmd], check=check)

def port_open(host='127.0.0.1', port=PORT, timeout=0.25):
    try:
        with socket.create_connection((host, port), timeout=timeout):
            return True
    except OSError:
        return False

os.chdir('/content')

# Clone YOUR repo (no key version)
sh(f'rm -rf {REPO_DIR}', check=False)
sh('git clone https://github.com/uzairaffliate-sketch/Unlimited-Voice-Cloning.git', check=True)
os.chdir(REPO_DIR)

print('=== Quick system checks ===')
sh('nvidia-smi || true', check=False)

print('\n=== Installing server requirements ===')
if Path('requirements-nvidia.txt').exists():
    sh('/content/bin/micromamba run -n cb311 pip install -U pip setuptools wheel', check=False)
    sh('/content/bin/micromamba run -n cb311 pip install -r requirements-nvidia.txt', check=False)
else:
    sh(
        '/content/bin/micromamba run -n cb311 pip install -U pip setuptools wheel && '
        '/content/bin/micromamba run -n cb311 pip install '
        'fastapi "uvicorn[standard]" pyyaml soundfile librosa safetensors '
        'python-multipart requests jinja2 watchdog aiofiles unidecode inflect tqdm '
        'pydub audiotsm praat-parselmouth',
        check=False
    )

sh("/content/bin/micromamba run -n cb311 pip install --no-deps --force-reinstall 'protobuf>=4.25.0'", check=False)

print('\n=== Applying watermarker patch ===')
SITE_PKG = '/root/.local/share/mamba/envs/cb311/lib/python3.11/site-packages'
CB_DIR = Path(SITE_PKG) / 'chatterbox'
SENTINEL = '# [patched: watermarker made optional]'
TARGET = 'self.watermarker = perth.PerthImplicitWatermarker()'
patched = 0
for fname in ['tts.py', 'tts_turbo.py', 'mtl_tts.py', 'vc.py']:
    fp = CB_DIR / fname
    if not fp.exists():
        continue
    content = fp.read_text(encoding='utf-8')
    if SENTINEL in content or TARGET not in content:
        continue
    lines = content.split('\n')
    new_lines = []
    for line in lines:
        if TARGET in line and line.lstrip().startswith('self.'):
            ind = line[:len(line) - len(line.lstrip())]
            new_lines.append(f'{ind}{SENTINEL}')
            new_lines.append(f'{ind}try:')
            new_lines.append(f'{ind}    self.watermarker = perth.PerthImplicitWatermarker()')
            new_lines.append(f'{ind}except Exception:')
            new_lines.append(f'{ind}    class _NoOpWatermarker:')
            new_lines.append(f'{ind}        def apply_watermark(self, wav, *args, **kwargs):')
            new_lines.append(f'{ind}            return wav')
            new_lines.append(f'{ind}    self.watermarker = _NoOpWatermarker()')
        else:
            new_lines.append(line)
    fp.write_text('\n'.join(new_lines), encoding='utf-8')
    print(f'  Patched {fname}')
    patched += 1
if patched:
    print(f'  {patched} file(s) patched')
else:
    print('  No patching needed')

Path(LOG_STDOUT).unlink(missing_ok=True)

print('\n=== Starting server ===')

env = os.environ.copy()
env['PYTHONUNBUFFERED'] = '1'
env['HF_HOME'] = '/content/hf_home'
env['TRANSFORMERS_CACHE'] = '/content/hf_home/transformers'
env['HF_HUB_CACHE'] = '/content/hf_home/hub'
Path(env['HF_HOME']).mkdir(parents=True, exist_ok=True)

proc = subprocess.Popen(
    ['/content/bin/micromamba', 'run', '-n', 'cb311', 'python', '-u', 'server.py'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    env=env,
)

with open(LOG_STDOUT, 'w', encoding='utf-8', errors='replace') as f:
    shown_link = False
    while True:
        line = proc.stdout.readline()
        if line:
            print(line, end='')
            f.write(line)
            f.flush()

        if (not shown_link) and port_open():
            shown_link = True
            from google.colab.output import eval_js
            proxy_url = eval_js(f'google.colab.kernel.proxyPort({PORT})')
            print(f'\n🌐 Open this URL: {proxy_url}\n')

        if proc.poll() is not None:
            print('\n=== Server process exited with code', proc.returncode, '===')
            break